# Data Mining Assignment 3 - Human Activity Recognition (HAR)

## Task Analysis
**Objective**: Predict each user's activity in 5-minute intervals using accelerometer readings.
- **Input**: 3-axis accelerometer readings (mean_x, mean_y, mean_z, std_x, std_y, std_z) recorded over a 5-minute period (300 seconds). Each row represents statistics for 1 second.
- **Output**: A single activity label (0-5) for each 5-minute time window.
- **Evaluation**: F1-score (macro).

Let's start by exploring the provided data!

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

sns.set_theme(style="whitegrid")

## 1. Data Loading (directly from CSVs)
To ensure this works perfectly on Kaggle, we load the raw `.csv` files from the competition dataset.

In [ ]:
# Pfad anpassen, falls nötig! Auf Kaggle meist: '/kaggle/input/nycu-data-mining-assignment-3'
KAGGLE_PATH = '/kaggle/input/nycu-data-mining-assignment-3'

# Fallback für lokales Ausführen
if not os.path.exists(KAGGLE_PATH):
    KAGGLE_PATH = 'nycu-data-mining-assignment-3'

def load_train_data(base_path):
    all_data = []
    all_labels = []
    all_file_ids = []
    all_user_ids = []
    
    # Suche nach den User-Ordnern im Trainings-Verzeichnis
    search_path = os.path.join(base_path, "train", "train", "User_*")
    user_dirs = sorted(glob.glob(search_path))
    
    if not user_dirs:
        # Fallback falls die Struktur leicht anders ist
        search_path = os.path.join(base_path, "train", "User_*")
        user_dirs = sorted(glob.glob(search_path))
    
    for user_dir in tqdm(user_dirs, desc="Loading train data"):
        user_id = os.path.basename(user_dir)
        csv_files = sorted(glob.glob(os.path.join(user_dir, "*.csv")))
        
        for csv_file in csv_files:
            df = pd.read_csv(csv_file)
            
            # Extrahiere die 6 Features
            features = df[['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z']].values
            label = df['label'].iloc[0]
            file_id = df['file_id'].iloc[0]
            
            all_data.append(features)
            all_labels.append(label)
            all_file_ids.append(file_id)
            all_user_ids.append(user_id)
            
    return np.array(all_data), np.array(all_labels), np.array(all_file_ids), np.array(all_user_ids)

X, y, file_ids, user_ids = load_train_data(KAGGLE_PATH)

print(f"X shape: {X.shape} (samples, time_steps, features)")
print(f"y shape: {y.shape} (labels)")
print(f"Number of unique users: {len(np.unique(user_ids))}")

## 2. Class Distribution
Let's check if our classes (activities 0-5) are balanced.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x=y, palette="viridis")
plt.title('Distribution of Activity Labels (0-5)')
plt.xlabel('Activity Label')
plt.ylabel('Count')
plt.show()

## 3. Visualizing Accelerometer Signals
Let's visualize one 5-minute sequence for each activity label to see how the acceleration patterns differ.

In [ ]:
features = ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z']

fig, axes = plt.subplots(6, 1, figsize=(12, 18), sharex=True)

for label in range(6):
    # Find the first index corresponding to the current label
    idx = np.where(y == label)[0][0]
    sample_sequence = X[idx]
    
    ax = axes[label]
    # Plotting mean_x, mean_y, mean_z
    ax.plot(sample_sequence[:, 0], label='mean_x', alpha=0.8)
    ax.plot(sample_sequence[:, 1], label='mean_y', alpha=0.8)
    ax.plot(sample_sequence[:, 2], label='mean_z', alpha=0.8)
    
    ax.set_title(f'Activity Label: {label}')
    ax.set_ylabel('Acceleration (g)')
    if label == 0:
        ax.legend(loc='upper right')

axes[-1].set_xlabel('Time Steps (seconds)')
plt.tight_layout()
plt.show()